# Data Cleaning Script

**Workforce Intelligence and Organizational Performance Analysis**  
Roblox Africa Operations (Ghana Hub)

Stage 5 — Data Cleaning &nbsp;·&nbsp; 31 July 2026

---

### What this notebook does

It turns the seven raw files into seven cleaned files, and proves the result is correct.

Run it top to bottom. It reads the raw data, applies every correction, checks its own work,
and writes the output. If any check fails the notebook stops rather than saving bad data.

### Why a script rather than cleaning by hand in Excel

The brief requires the project to be *reproducible and well documented*. Cleaning by hand in
Excel satisfies neither: nobody can see what was done, and nobody can repeat it on next
month's extract. This notebook can be re-run by anyone, on new data, and will do exactly the
same thing every time.

### The one rule

**The raw files are never modified.** They are opened read-only. Every change happens in memory
and is written to a separate output folder. If anything goes wrong, re-run from the top.

### Where the corrections come from

| Source | What it provides |
|---|---|
| `TIFC - DA - SAC3 - Mistakes Handout` | The 21 logged errors |
| `data_quality_summary.docx` | Which errors we fix by rule, which need the client |
| `client_decisions_log.docx` | The 8 values the client confirmed, plus the USD instruction |
| `cleaning_plan.xlsx` | The order of work, and the reason behind each step |

## 1 &nbsp; Setup

`RAW` is read-only. `OUT` is where the cleaned files are written.

In [ ]:
import pandas as pd
import numpy as np
import os

RAW = "../01_Raw_Data"      # the seven original files - never written to
OUT = "../04_Clean_Data"    # where the cleaned files go
os.makedirs(OUT, exist_ok=True)

# every change gets recorded here as we go, never afterwards from memory
log = []

def record(dataset, action, detail, before, after):
    """Add one line to the cleaning log."""
    log.append({"Dataset": dataset, "Action": action, "Detail": detail,
                "Before": before, "After": after})
    print(f"  {dataset:<24} {action:<38} {before}  ->  {after}")

print("ready")

## 2 &nbsp; Load the raw data

Nothing is changed here. We only read, and note the starting row counts so that every later
change can be measured against them.

In [ ]:
emp  = pd.read_excel(f"{RAW}/employee_dataset.xlsx")
dept = pd.read_excel(f"{RAW}/Department.xlsx")
edu  = pd.read_excel(f"{RAW}/Education.xlsx")
hea  = pd.read_excel(f"{RAW}/health_dataset.xlsx")
fin  = pd.read_excel(f"{RAW}/finance_dataset.xlsx")
epf  = pd.read_excel(f"{RAW}/employee_performance.xlsx")
dpf  = pd.read_excel(f"{RAW}/department_performance.xlsx")

RAW_ROWS = {
    "employee_dataset": len(emp), "Department": len(dept), "Education": len(edu),
    "health_dataset": len(hea), "finance_dataset": len(fin),
    "employee_performance": len(epf), "department_performance": len(dpf),
}
for k, v in RAW_ROWS.items():
    print(f"{k:<24} {v:>8,} rows")
print(f"{chr(10)}total{chr(10)}{sum(RAW_ROWS.values()):>32,} rows")

## 3 &nbsp; Step 1 — Remove duplicate rows

**This must come first.**

If we corrected an age and *then* removed duplicates, we might have done the same correction
twice for nothing. Worse, every row count checked along the way would have been measured
against a total that was wrong.

`drop_duplicates()` with no arguments removes rows that are identical in **every** column. It
will not touch two different people who happen to share a surname.

In [ ]:
n = len(emp)
emp = emp.drop_duplicates().reset_index(drop=True)
record("employee_dataset", "Removed duplicate rows",
       "4 rows were exact copies (EmployeeID 14813, 18044, 29979, 29980)", n, len(emp))

n = len(edu)
edu = edu.drop_duplicates().reset_index(drop=True)
record("Education", "Removed duplicate row",
       "1 row was an exact copy (employee_id 1)", n, len(edu))

## 4 &nbsp; Step 2 — Apply the values the client confirmed

Every value below comes from the client's reply to `Roblox_Data_Error_Report_v1`, and is
recorded in the Client Decisions Log. The reference codes (C1, C2, …) point at the row in
that log which authorises each change.

Nothing here is guessed. Where the client did not answer, we do not act — see section 9.

In [ ]:
# ---- C1, C2 : two impossible ages ----
before = f"min age {emp['Age'].min()}"
emp.loc[emp["EmployeeID"] == 279,   "Age"] = 48    # was -48, a sign error
emp.loc[emp["EmployeeID"] == 21187, "Age"] = 30    # was 0
record("employee_dataset", "Corrected invalid ages",
       "ID 279 -> 48; ID 21187 -> 30. Client authority C1, C2",
       before, f"min age {emp['Age'].min()}")

In [ ]:
# ---- C3 : seven missing genders ----
n = int(emp["Gender"].isna().sum())
emp["Gender"] = emp["Gender"].fillna("Female")
record("employee_dataset", "Filled missing Gender",
       "7 blanks -> Female (IDs 4470, 5826, 9301, 12333, 19342, 25405, 26938). Authority C3",
       f"{n} blank", f"{int(emp['Gender'].isna().sum())} blank")

# ---- C6 : seven missing employment statuses ----
n = int(emp["employee_status"].isna().sum())
emp["employee_status"] = emp["employee_status"].fillna("Active")
record("employee_dataset", "Filled missing employee_status",
       "7 blanks -> Active (IDs 1224, 6765, 7700, 13921, 21913, 22187, 25999). Authority C8",
       f"{n} blank", f"{int(emp['employee_status'].isna().sum())} blank")

In [ ]:
# ---- C6 : one missing surname ----
# The email address on the same row reads Kwame.Sebastian.Mohammed.Campbell@roblox.com,
# which independently corroborates the client's answer.
emp.loc[emp["EmployeeID"] == 28686, "Last Name"] = "Campbell"
record("employee_dataset", "Filled missing Last Name",
       "ID 28686 -> Campbell. Authority C6", "1 blank", "0 blank")

# ---- C5 : one missing department code ----
# Without this the employee is silently excluded from every department total.
emp.loc[emp["EmployeeID"] == 25142, "Department Code"] = "FIN"
record("employee_dataset", "Filled missing Department Code",
       "ID 25142 -> FIN. Authority C5", "1 blank", "0 blank")

## 5 &nbsp; Step 3 — Fix the spelling variant

One record reads `Femal` instead of `Female`. The cell is **not blank**, so the missing-value
fill in the previous cell never touched it.

`.replace()` on a whole column matches the entire cell value. This is the safe behaviour: a
find-and-replace on the *text* would turn every `Female` into `Femalee`.

In [ ]:
n = int((emp["Gender"] == "Femal").sum())
emp["Gender"] = emp["Gender"].replace("Femal", "Female")   # whole-cell match
record("employee_dataset", "Corrected Gender spelling",
       "ID 21222 'Femal' -> 'Female'. Authority C4",
       f"{n} row", f"{int((emp['Gender'] == 'Femal').sum())} rows")

## 6 &nbsp; Step 4 — Convert `Date Joined` into a real date

The column holds numbers like `45081`. That is Excel's internal date code — the count of days
since 30 December 1899. Until it is converted, no tenure or length-of-service question can be
answered at all.

`origin='1899-12-30'` is not a typo. Excel incorrectly treats 1900 as a leap year, and this
origin compensates for it.

In [ ]:
emp["Date Joined"] = pd.to_datetime(emp["Date Joined"], unit="D", origin="1899-12-30")
record("employee_dataset", "Converted Date Joined to a date",
       "Excel serial numbers converted using origin 1899-12-30",
       "integer 42074-45724",
       f"{emp['Date Joined'].min().date()} to {emp['Date Joined'].max().date()}")

## 7 &nbsp; Step 5 — Identifiers become text

**Order matters here.** The seven missing phone numbers are filled *first*, then the whole
column is converted. Convert first and those seven empty cells become the literal string
`'nan'` — a new problem created while fixing an old one.

A phone number is a label, not a quantity. Nobody adds two phone numbers together. Stored as a
number it can lose a leading digit and display as `2.33E+11`. The same reasoning applies to
bank account numbers.

In [ ]:
# the seven correct numbers, supplied by the client (Decisions Log section B)
PHONE_FIX = {
    3257: 233590129809, 14367: 233556780467, 17803: 233500129809,
    17986: 233500012987, 21281: 233260125678, 23486: 233530895670,
    28940: 233540122309,
}

n  = int(emp["Phone Number"].isna().sum())
ph = emp["Phone Number"]

# fill the blanks, THEN convert the whole column to text
emp["Phone Number"] = (ph.where(ph.notna(), emp["EmployeeID"].map(PHONE_FIX))
                         .astype("int64").astype(str))

record("employee_dataset", "Filled and converted Phone Number",
       "7 numbers supplied by the client (authority C7), then column converted to text",
       f"{n} blank, stored as number", "0 blank, stored as text")

In [ ]:
# Account numbers: values are NOT changed, only the storage type.
# Lengths vary from 5 to 13 digits. That may be legitimate across ten banks,
# so it stays an open question (O3) rather than a correction.
fin["Account Number"] = fin["Account Number"].astype("int64").astype(str)
record("finance_dataset", "Converted Account Number to text",
       "Values unchanged. Length still varies 5-13 digits - open question O3",
       "stored as number", "stored as text, values unchanged")

## 8 &nbsp; Step 6 — Tidy text, rename, and drop a redundant column

`' Analyst'` and `'Analyst'` look identical on screen but count as two different job titles.
Trimming stray spaces prevents a category silently splitting in two.

Note the rename is **targeted**. Lowercasing every header would have turned `Employee ID` into
`employee id`, breaking the join key. Only the one inconsistent column is renamed.

In [ ]:
def trim_text(df):
    """Remove leading and trailing spaces from every text column."""
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            df[c] = df[c].astype(str).str.strip()
    return df

for name, df in [
    ("employee_dataset", emp), ("Department", dept), ("Education", edu),
    ("health_dataset", hea), ("finance_dataset", fin),
    ("employee_performance", epf), ("department_performance", dpf),
]:
    trim_text(df)

record("all seven", "Trimmed stray spaces from text columns",
       "Prevents categories splitting in two", "-", "no stray spaces")

In [ ]:
# created_at and Updated_at differed only in capitalisation.
# Harmless in Excel; SQL and Python will refuse to find the column.
hea = hea.rename(columns={"Updated_at": "updated_at"})
record("health_dataset", "Renamed Updated_at to updated_at",
       "Only this column renamed - lowercasing all headers would break 'Employee ID'",
       "Updated_at", "updated_at")

# Education Record_id was identical to employee_id in all 30,000 rows.
# Two columns saying the same thing invites joining on the wrong one.
edu = edu.drop(columns=["Education Record_id"])
record("Education", "Dropped Education Record_id",
       "Duplicated employee_id in every row. The raw file retains it",
       "8 columns", "7 columns")

## 9 &nbsp; What we deliberately did **not** change

Knowing when not to clean is the harder half of the job. Changing any of the following would
mean inventing information we do not have.

| Left alone | Why | Reference |
|---|---|---|
| 5 records reading `Information Technology & Law` | 4,326 read `Information Technology`. Probably a variant, but changing it would rewrite somebody's qualification | O4 |
| Account numbers of 5 to 13 digits | May be legitimate across ten different banks. Inventing digits is far worse than reporting the inconsistency | O3 |
| Salary period (monthly or annual?) | Never stated in the source. Multiplying by 12 on an assumption could be wrong by twelve times | O1 |
| `Total_Cost` not reconciling with payroll | The two tables are on different scales. Neither is 'wrong'; they measure different things | O2 |
| 7,193 employees who joined before graduating | Studying while employed is normal. Not an error | O5 |

Each is recorded as an open question in the Client Decisions Log, with a documented assumption
to apply if no answer arrives before the deadline.

## 10 &nbsp; Verification

Cleaning is not finished when the code runs without an error. It is finished when the output
is *proved* correct.

Two kinds of check below:

- **Did the fixes land?** — row counts, no blanks, the specific corrected records
- **Did anything change that shouldn't have?** — totals compared against the raw files

That second kind matters most. It is how you catch a cleaning step that quietly damaged data
it was never meant to touch. Note that these compare against the **raw files**, not against
numbers typed from memory.

In [ ]:
checks = []

def check(desc, got, expect):
    ok = (got == expect)
    checks.append({"Check": desc, "Expected": str(expect),
                   "Got": str(got), "Result": "PASS" if ok else "FAIL"})

# ---- did the fixes land? ----
check("employee_dataset row count", len(emp), 30000)
check("employee_dataset blank cells", int(emp.isna().sum().sum()), 0)
check("distinct Gender values", sorted(emp["Gender"].unique()), ["Female", "Male"])
check("minimum Age at least 18", int(emp["Age"].min()) >= 18, True)
check("maximum Age at most 65", int(emp["Age"].max()) <= 65, True)
check("Date Joined is a date type", str(emp["Date Joined"].dtype).startswith("datetime"), True)
check("Phone Number is text", pd.api.types.is_string_dtype(emp["Phone Number"]), True)
check("all phones are 12 digits", emp["Phone Number"].str.len().eq(12).all(), True)
check("all phones start 233", emp["Phone Number"].str.startswith("233").all(), True)
check("ID 279 age", int(emp.loc[emp.EmployeeID == 279, "Age"].iloc[0]), 48)
check("ID 21187 age", int(emp.loc[emp.EmployeeID == 21187, "Age"].iloc[0]), 30)
check("ID 25142 department", emp.loc[emp.EmployeeID == 25142, "Department Code"].iloc[0], "FIN")
check("ID 28686 surname", emp.loc[emp.EmployeeID == 28686, "Last Name"].iloc[0], "Campbell")
check("ID 21222 gender", emp.loc[emp.EmployeeID == 21222, "Gender"].iloc[0], "Female")
check("Education row count", len(edu), 30000)
check("Department row count", len(dept), 8)
check("health_dataset row count", len(hea), 30000)
check("finance_dataset row count", len(fin), 30000)
check("employee_performance row count", len(epf), 150000)
check("department_performance row count", len(dpf), 40)
check("Account Number is text", pd.api.types.is_string_dtype(fin["Account Number"]), True)

In [ ]:
# ---- do the relationships still hold? ----
ids = set(emp["EmployeeID"])
check("Education keys all valid",  edu["employee_id"].isin(ids).all(), True)
check("finance keys all valid",    fin["StaffID"].isin(ids).all(), True)
check("health keys all valid",     hea["Employee ID"].isin(ids).all(), True)
check("performance keys all valid",epf["EmployeeID"].isin(ids).all(), True)
check("department codes all valid",
      emp["Department Code"].isin(set(dept["department_code"])).all(), True)

# ---- did anything change that shouldn't have? compare to the RAW files ----
raw_fin = pd.read_excel(f"{RAW}/finance_dataset.xlsx")
raw_epf = pd.read_excel(f"{RAW}/employee_performance.xlsx")
raw_dpf = pd.read_excel(f"{RAW}/department_performance.xlsx")

check("Basic Salary total unchanged", int(fin["Basic Salary"].sum()),
      int(raw_fin["Basic Salary"].sum()))
check("Allowances total unchanged", int(fin["Allowances"].sum()),
      int(raw_fin["Allowances"].sum()))
check("performance scores unchanged", round(float(epf["Performance_Score"].sum()), 2),
      round(float(raw_epf["Performance_Score"].sum()), 2))
check("department revenue unchanged", int(dpf["Total_Revenue_Generated"].sum()),
      int(raw_dpf["Total_Revenue_Generated"].sum()))
check("department cost unchanged", int(dpf["Total_Cost"].sum()),
      int(raw_dpf["Total_Cost"].sum()))

In [ ]:
results = pd.DataFrame(checks)
failed  = results[results["Result"] == "FAIL"]

print(results.to_string(index=False))
print(f"{chr(10)}{len(results) - len(failed)} of {len(results)} checks passed")

if len(failed):
    print(f"{chr(10)}FAILED CHECKS:")
    print(failed.to_string(index=False))
    raise SystemExit("Cleaning is not complete. Fix the failures above before saving.")

print(f"{chr(10)}All checks passed. Safe to save.")

## 11 &nbsp; Save the cleaned data

Two formats, because they serve different purposes:

- **Seven separate files** — what Stage 6 loads into SQL, one table at a time
- **One workbook** — easy to review and share, with the log and verification included

The cleaning log and the verification results are saved *alongside* the data. A cleaned file
without its log is just a file somebody says is clean.

In [ ]:
DATA = {
    "employee_dataset": emp, "Department": dept, "Education": edu,
    "health_dataset": hea, "finance_dataset": fin,
    "employee_performance": epf, "department_performance": dpf,
}

# --- seven individual files ---
for name, df in DATA.items():
    df.to_excel(f"{OUT}/{name}_clean.xlsx", index=False)
    print(f"  saved {name}_clean.xlsx  ({len(df):,} rows)")

In [ ]:
# --- one consolidated workbook, with the log and the checks ---
log_df = pd.DataFrame(log)
summary = pd.DataFrame([{
    "Dataset": k,
    "Rows before cleaning": RAW_ROWS[k],
    "Rows after cleaning": len(v),
    "Columns": len(v.columns),
    "Changes made": int((log_df["Dataset"] == k).sum()),
} for k, v in DATA.items()])

with pd.ExcelWriter(f"{OUT}/cleaned_datasets.xlsx", engine="openpyxl") as xw:
    summary.to_excel(xw, sheet_name="SUMMARY", index=False)
    log_df.to_excel(xw,  sheet_name="CLEANING LOG", index=False)
    results.to_excel(xw, sheet_name="VERIFICATION", index=False)
    for name, df in DATA.items():
        df.to_excel(xw, sheet_name=name, index=False)

print(f"{chr(10)}saved cleaned_datasets.xlsx")

In [ ]:
# --- ALWAYS read back what you just wrote ---
# On an earlier run a formatting bug silently emptied two sheets. It was only
# caught by reopening the file. Writing successfully is not the same as saving correctly.

check_back = pd.read_excel(f"{OUT}/cleaned_datasets.xlsx", sheet_name="employee_dataset")
print(f"read back: {len(check_back):,} rows, {int(check_back.isna().sum().sum())} blanks")

assert len(check_back) == 30000, "the saved file does not contain 30,000 rows"
print("verified on disk")

---

## Summary

| | |
|---|---|
| Changes made | 14, every one logged with its authority |
| Verification checks | 31, all passing |
| Rows in → out | 270,053 → 270,048 (5 duplicates removed) |
| Values invented | none |

### What comes next

Stage 6 loads these seven files into a SQL database and joins the five person-level tables into
one master employee record. The two performance tables stay separate — merging them would
multiply every salary by five.

**The test that matters in Stage 6:** after joining, the master table must contain exactly
**30,000** rows. Any more and a join is duplicating.